In [2]:
import pandas as pd
from bs4 import BeautifulSoup
import numpy as np

In [3]:
with open("SEPTUAGINT.xml") as f:
    sept_soup = BeautifulSoup(f.read())

with open("TISCHENDORF.xml") as file:
    tisch_soup = BeautifulSoup(file)


rm_punct_tbl = str.maketrans("", "", " .·,:;!()«»-·")
sept = pd.DataFrame(
    [
        (
            str(e.text).translate(rm_punct_tbl).lower(),
            e.get("str", "G").removeprefix("G"),
            e.find_parent("vers")["vnumber"],
            e.find_parent("chapter")["cnumber"],
            e.find_parent("biblebook")["bnumber"],
            e["rmac"],
        )
        for e in sept_soup.find_all("gr")
    ],
    columns=["text", "str", "verse", "chapter", "book", "rmac"],
)
tisch = pd.DataFrame(
    [
        (
            str(e.text).translate(rm_punct_tbl).lower(),
            e.get("str"),
            e.find_parent("vers")["vnumber"],
            e.find_parent("chapter")["cnumber"],
            e.find_parent("biblebook")["bnumber"],
            e["rmac"],
        )
        for e in tisch_soup.find_all("gr")
    ],
    columns=["text", "str", "verse", "chapter", "book", "rmac"],
)

/tmp/ipykernel_8786/2983391643.py:2: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  sept_soup = BeautifulSoup(f.read())
/tmp/ipykernel_8786/2983391643.py:5: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, b

In [4]:
sept["verse"] = sept["verse"].astype(int)
sept["chapter"] = sept["chapter"].astype(int)
sept["book"] = sept["book"].astype(int)
# include code to parse rmac?

In [5]:
tisch["verse"] = tisch["verse"].astype(int)
tisch["chapter"] = tisch["chapter"].astype(int)
tisch["book"] = tisch["book"].astype(int)

## Reconciling missing strongs

In [22]:
np.arange(len(no_str_words)).astype(str)

array(['0', '1', '2', ..., '13670', '13671', '13672'], dtype='<U21')

In [23]:
'C' + np.arange(len(no_str_words)).astype(str)


UFuncTypeError: ufunc 'add' did not contain a loop with signature matching types (dtype('<U1'), dtype('<U21')) -> None

In [25]:
['C' + str(number) for number in range(0, len(no_str_words))]

['C0',
 'C1',
 'C2',
 'C3',
 'C4',
 'C5',
 'C6',
 'C7',
 'C8',
 'C9',
 'C10',
 'C11',
 'C12',
 'C13',
 'C14',
 'C15',
 'C16',
 'C17',
 'C18',
 'C19',
 'C20',
 'C21',
 'C22',
 'C23',
 'C24',
 'C25',
 'C26',
 'C27',
 'C28',
 'C29',
 'C30',
 'C31',
 'C32',
 'C33',
 'C34',
 'C35',
 'C36',
 'C37',
 'C38',
 'C39',
 'C40',
 'C41',
 'C42',
 'C43',
 'C44',
 'C45',
 'C46',
 'C47',
 'C48',
 'C49',
 'C50',
 'C51',
 'C52',
 'C53',
 'C54',
 'C55',
 'C56',
 'C57',
 'C58',
 'C59',
 'C60',
 'C61',
 'C62',
 'C63',
 'C64',
 'C65',
 'C66',
 'C67',
 'C68',
 'C69',
 'C70',
 'C71',
 'C72',
 'C73',
 'C74',
 'C75',
 'C76',
 'C77',
 'C78',
 'C79',
 'C80',
 'C81',
 'C82',
 'C83',
 'C84',
 'C85',
 'C86',
 'C87',
 'C88',
 'C89',
 'C90',
 'C91',
 'C92',
 'C93',
 'C94',
 'C95',
 'C96',
 'C97',
 'C98',
 'C99',
 'C100',
 'C101',
 'C102',
 'C103',
 'C104',
 'C105',
 'C106',
 'C107',
 'C108',
 'C109',
 'C110',
 'C111',
 'C112',
 'C113',
 'C114',
 'C115',
 'C116',
 'C117',
 'C118',
 'C119',
 'C120',
 'C121',
 'C122',
 'C

In [26]:
# words without strongs
nameless = pd.concat(
    [tisch[~tisch["str"].astype(bool)], sept[~sept["str"].astype(bool)]]
)

# unique
no_str_words = pd.Series(nameless["text"].unique())


new_strongs = pd.Series(
    ['C' + str(number) for number in range(0, len(no_str_words))],
    index=no_str_words
)

In [ ]:
tisch_needs_new = ~tisch["str"].astype(
    bool
)  # if null or "" sets to False, anything else -> True.
tisch.loc[tisch_needs_new, "str"] = new_strongs.loc[
    tisch.loc[tisch_needs_new, "text"]
].values

sept_needs_new = ~sept["str"].astype(bool)
sept.loc[sept_needs_new, "str"] = new_strongs.loc[
    sept.loc[sept_needs_new, "text"]
].values

In [ ]:
tisch.to_pickle('./pickles/tisch.pickle')
sept.to_pickle('./pickles/sept.pickle')